## 34 — Promotional Features: DTC (Direct Buy + Social)

Generates DMA-level ad impressions and clicks from Direct Buy vendors and Meta/IG social.
No rolling windows — raw monthly aggregates (Variant C).  
**Source:** Non-personal skill §6 + §7.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, lit, upper

In [0]:
BASE_VOLUME = (
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
    "kerendia_nba_3_volume/hcp_level_model"
)

# Source data (ops pipeline outputs)
DIRECT_BUYS_PATH = f"{BASE_VOLUME}/ops_pipeline/query_output/kerendia_nba_3_0_hcp_model_DirectBuys"
SOCIAL_PATH = f"{BASE_VOLUME}/ops_pipeline/query_output/kerendia_nba_3_0_hcp_model_social"

# Output
OUTPUT_PATH = f"{BASE_VOLUME}/dev_folder/notebook_output/kerendia_nba_3_hcp_model_DTC_PROMOTION.parquet"

In [0]:
def sanitize_dtc(name):
    """DTC-specific column sanitization."""
    return (name.replace(", ", " ").replace("+", "and").replace("/", " or ")
            .replace("-", "_").replace(" ", "_").replace("__", "_")
            .replace("(", "").replace(")", "").replace("&", "").upper())

In [0]:
# Single consolidated Delta table for all 4 vendors
df_db = spark.read.format("delta").load(DIRECT_BUYS_PATH)
print(f"Direct Buy data: {df_db.count()} rows")
print(f"Columns: {df_db.columns}")
df_db.show(5)

# Show vendors
df_db.select("VENDOR").distinct().show()

In [0]:
# Rename MONTH -> DATA_MONTH
df_db = df_db.withColumnRenamed("MONTH", "DATA_MONTH")

# Pivot by vendor: impressions and clicks per DMA x month
vendors = [row.VENDOR for row in df_db.select("VENDOR").distinct().collect()]

df_db_pivot = df_db.groupBy("DMA_Code", "DATA_MONTH").pivot("VENDOR", vendors).agg(
    F.sum("IMPRESSIONS").alias("TOTAL_IMPRESSIONS"),
    F.sum("CLICKS").alias("TOTAL_CLICKS")
).fillna(0)

# Sanitize column names and add DTC_PROMOTION_ prefix
for c in df_db_pivot.columns:
    if c not in ["DMA_Code", "DATA_MONTH"]:
        new_name = f"DTC_PROMOTION_{sanitize_dtc(c)}"
        df_db_pivot = df_db_pivot.withColumnRenamed(c, new_name)

print(f"Direct Buy pivoted: {df_db_pivot.count()} rows, {len(df_db_pivot.columns)} columns")
df_db_pivot.show(5)

In [0]:
# Social Delta table (FB+IG)
df_social = spark.read.format("delta").load(SOCIAL_PATH)
df_social = df_social.withColumnRenamed("MONTH", "DATA_MONTH")
print(f"Social data: {df_social.count()} rows")
df_social.show(5)

In [0]:
vendors_social = [row.VENDOR for row in df_social.select("VENDOR").distinct().collect()]

df_social_pivot = df_social.groupBy("DMA_Code", "DATA_MONTH").pivot("VENDOR", vendors_social).agg(
    F.sum("IMPRESSIONS").alias("TOTAL_IMPRESSIONS"),
    F.sum("CLICKS").alias("TOTAL_CLICKS")
).fillna(0)

# Sanitize and prefix
for c in df_social_pivot.columns:
    if c not in ["DMA_Code", "DATA_MONTH"]:
        new_name = f"DTC_PROMOTION_{sanitize_dtc(c)}"
        df_social_pivot = df_social_pivot.withColumnRenamed(c, new_name)

print(f"Social pivoted: {df_social_pivot.count()} rows, {len(df_social_pivot.columns)} columns")

In [0]:
# Outer join on DMA_Code + DATA_MONTH
df_dtc = df_db_pivot.join(df_social_pivot, on=["DMA_Code", "DATA_MONTH"], how="outer").fillna(0)

print(f"Combined DTC output: {df_dtc.count()} rows, {len(df_dtc.columns)} columns")
print(f"Columns: {df_dtc.columns}")
df_dtc.show(5)

In [0]:
df_dtc.write.format("parquet").mode("overwrite").save(OUTPUT_PATH)
print(f"Written to: {OUTPUT_PATH}")